# 🌍 WHO EMRO Health Indicators Analysis

This notebook demonstrates analysis of health indicators for the **WHO Eastern Mediterranean Region (EMRO)**.

## EMRO Region Overview

The Eastern Mediterranean Region includes **22 countries** spanning:
- Middle East (Jordan, Lebanon, Syria, Iraq, Iran)
- Arabian Peninsula (Saudi Arabia, UAE, Qatar, Kuwait, Bahrain, Oman, Yemen)
- North Africa (Egypt, Libya, Tunisia, Algeria, Morocco, Sudan, Somalia, Djibouti)
- Central/South Asia (Afghanistan, Pakistan, Palestine)

**Key Health Challenges:**
- Malaria (Afghanistan, Pakistan, Somalia, Sudan, Yemen)
- Malnutrition and food insecurity
- Conflict-related health crises
- Non-communicable diseases
- Healthcare access disparities

**Requirements:**
```bash
pip install ghoclient pandas matplotlib seaborn plotly
```

## 1. Setup and Imports

In [ ]:
import sys
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import WHO accessor
from epidatasets.sources.who_ghoclient import WHOAccessor, EMRO_COUNTRIES

print("✅ Imports completed successfully!")
print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"📊 Analyzing WHO EMRO region")

## 2. Initialize WHO Accessor and Explore EMRO

In [ ]:
# Initialize WHO accessor
who = WHOAccessor()

print("🌍 WHO Global Health Observatory Accessor initialized")

# Get EMRO countries
emro_countries = who.get_emro_countries()
print(f"\n🇺🇳 EMRO Countries ({len(emro_countries)} total):")
print(emro_countries.to_string(index=False))

## 3. Visualize EMRO Countries

In [ ]:
# Create visualization of EMRO countries
fig, ax = plt.subplots(figsize=(12, 10))

y_pos = range(len(emro_countries))
colors = plt.cm.Spectral(np.linspace(0, 1, len(emro_countries)))

bars = ax.barh(y_pos, [1]*len(emro_countries), color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels(emro_countries['country_name'], fontsize=9)
ax.set_xlabel('Countries')
ax.set_title('WHO Eastern Mediterranean Region (EMRO)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.set_xlim(0, 1.2)

plt.tight_layout()
plt.show()

print("\n📊 EMRO Sub-regions (22 countries):")
print("  • Middle East: Lebanon, Syria, Jordan, Iraq, Iran (5)")
print("  • Arabian Peninsula: Saudi Arabia, UAE, Qatar, Kuwait, Bahrain, Oman, Yemen (7)")
print("  • North Africa: Egypt, Libya, Tunisia, Algeria, Morocco, Sudan, Somalia, Djibouti (8)")
print("  • Central/South Asia: Afghanistan, Pakistan, Palestine (3)")

## 4. Search for Health Indicators

In [ ]:
# Search for key health indicators relevant to EMRO
search_terms = [
    "life expectancy",
    "malaria",
    "maternal mortality",
    "child mortality"
]

print("🔍 Searching for key health indicators...")

for term in search_terms:
    try:
        results = who.search_indicators(term)
        if not results.empty:
            print(f"\n{term.upper()}:")
            print(results[['IndicatorCode', 'IndicatorName']].head(2).to_string(index=False))
    except Exception as e:
        print(f"  ⚠️ Could not search {term}: {e}")

print("\n✅ Indicator search complete")

## 5. Key EMRO Health Indicators

In [ ]:
# Define key indicators for EMRO analysis
key_indicators = {
    "WHOSIS_000001": "Life expectancy at birth",
    "WHOSIS_000002": "Healthy life expectancy (HALE)",
    "MDG_0000000001": "Under-five mortality rate",
    "MDG_0000000025": "Maternal mortality ratio",
    "MALARIA_EST_INCIDENCE": "Malaria incidence per 1,000",
    "TB_e_inc_num": "TB incidence",
    "WHS4_129": "DPT immunization coverage"
}

print("📊 Key Health Indicators for EMRO Analysis:")
print("="*60)
for code, name in key_indicators.items():
    print(f"  {code}")
    print(f"    → {name}")

analysis_years = [2015, 2017, 2019, 2021]
print(f"\n📅 Analysis period: {min(analysis_years)}-{max(analysis_years)}")

## 6. Fetch Life Expectancy Data for EMRO

In [ ]:
# Fetch healthy life expectancy (HALE) for EMRO
print("📈 Fetching Healthy Life Expectancy (HALE) data for EMRO...")

try:
    hale_data = who.get_emro_life_expectancy(years=analysis_years)
    
    if not hale_data.empty:
        print(f"✅ Retrieved {len(hale_data)} records")
        print("\n🔍 Sample data:")
        print(hale_data[['country_code', 'year', 'value']].head(10))
        
        # Add country names
        hale_data = hale_data.merge(emro_countries[['country_code', 'country_name']], on='country_code')
    else:
        print("⚠️ No data retrieved")
except Exception as e:
    print(f"⚠️ Could not fetch HALE data: {e}")
    hale_data = pd.DataFrame()

## 7. Visualize HALE Trends

In [ ]:
# Visualize HALE trends across EMRO
if not hale_data.empty:
    # Get EMRO regional trends
    emro_trends = who.get_emro_health_trends('WHOSIS_000002', 2015, 2021)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: EMRO regional average
    if not emro_trends.empty:
        ax1.plot(emro_trends['year'], emro_trends['emro_mean'],
                marker='o', linewidth=3, markersize=8, label='EMRO Average')
        ax1.fill_between(emro_trends['year'],
                        emro_trends['emro_min'],
                        emro_trends['emro_max'],
                        alpha=0.3, label='Min-Max Range')
        ax1.set_xlabel('Year', fontsize=12)
        ax1.set_ylabel('HALE (years)', fontsize=12)
        ax1.set_title('EMRO: Healthy Life Expectancy Trends', fontsize=14, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
    # Plot 2: Top and bottom countries
    latest_year = hale_data['year'].max()
    latest_data = hale_data[hale_data['year'] == latest_year].sort_values('value')
    
    # Show top 5 and bottom 5
    display_countries = pd.concat([latest_data.head(5), latest_data.tail(5)])
    colors = ['#e74c3c']*5 + ['#27ae60']*5
    
    ax2.barh(display_countries['country_name'], display_countries['value'], color=colors)
    ax2.set_xlabel('HALE (years)', fontsize=12)
    ax2.set_title(f'HALE by Country ({latest_year})', fontsize=14, fontweight='bold')
    ax2.axvline(x=latest_data['value'].mean(), color='blue', linestyle='--',
               label=f'EMRO Average: {latest_data["value"].mean():.1f}')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print(f"\n📊 HALE Summary for EMRO ({latest_year}):")
    print(f"  • Regional average: {latest_data['value'].mean():.1f} years")
    print(f"  • Highest: {latest_data.iloc[-1]['country_name']} ({latest_data.iloc[-1]['value']:.1f} years)")
    print(f"  • Lowest: {latest_data.iloc[0]['country_name']} ({latest_data.iloc[0]['value']:.1f} years)")
    print(f"  • Range: {latest_data['value'].max() - latest_data['value'].min():.1f} years")
else:
    print("⚠️ No HALE data available for visualization")

## 8. Malaria Analysis for EMRO

In [ ]:
# Malaria is endemic in parts of EMRO (Afghanistan, Pakistan, Somalia, Sudan, Yemen)
print("🦟 Fetching malaria incidence data for EMRO...")

try:
    malaria_data = who.get_emro_malaria_data(years=[2018, 2019, 2020])
    
    if not malaria_data.empty:
        print(f"✅ Retrieved {len(malaria_data)} records")
        
        # Add country names
        malaria_data = malaria_data.merge(
            emro_countries[['country_code', 'country_name']],
            on='country_code'
        )
        
        # Focus on countries with malaria burden
        high_burden = malaria_data[malaria_data['value'] > 0].sort_values('value', ascending=False)
        
        if not high_burden.empty:
            print("\n🔍 Malaria-endemic countries in EMRO:")
            print(high_burden[['country_name', 'year', 'value']].head(10))
        else:
            print("\n✓ Low malaria burden in EMRO (good progress!)")
    else:
        print("⚠️ No malaria data retrieved")
        malaria_data = pd.DataFrame()
except Exception as e:
    print(f"⚠️ Could not fetch malaria data: {e}")
    malaria_data = pd.DataFrame()

## 9. Compare EMRO Countries Across Indicators

In [ ]:
# Compare countries across multiple indicators
comparison_indicators = [
    ("WHOSIS_000001", "Life Expectancy"),
    ("MDG_0000000001", "Under-5 Mortality"),
    ("MDG_0000000025", "Maternal Mortality")
]

print("📊 Comparing EMRO countries across health indicators...")

comparison_results = {}
for code, name in comparison_indicators:
    try:
        data = who.get_emro_indicator(code, years=[2019])
        if not data.empty:
            comparison_results[name] = data
            print(f"✅ {name}: {len(data)} records")
    except Exception as e:
        print(f"⚠️ {name}: {e}")

print(f"\n📈 Retrieved {len(comparison_results)} indicators for comparison")

## 10. Create Health Dashboard

In [ ]:
# Create a comprehensive health dashboard
print("="*70)
print("🌍 WHO EMRO HEALTH DASHBOARD")
print("="*70)

print("\n📊 REGION OVERVIEW:")
print(f"  • Member countries: {len(emro_countries)}")
print(f"  • Population: ~583 million (2019 est.)")
print(f"  • Geographic coverage: Middle East, North Africa, Central Asia")

print("\n🏥 KEY HEALTH CHALLENGES:")
print("  1. Malaria: Endemic in 5 countries (AFG, PAK, SOM, SDN, YEM)")
print("  2. Conflict: Syria, Yemen, Sudan affecting health systems")
print("  3. NCDs: Rising burden of diabetes, cardiovascular disease")
print("  4. Immunization: Coverage gaps in conflict-affected areas")

if not hale_data.empty:
    latest_year = hale_data['year'].max()
    latest_hale = hale_data[hale_data['year'] == latest_year]
    print(f"\n📈 HEALTHY LIFE EXPECTANCY ({latest_year}):")
    print(f"  • EMRO Average: {latest_hale['value'].mean():.1f} years")
    print(f"  • Global comparison: ~2-3 years below global average")
    print(f"  • Best performers: Gulf States (UAE, Qatar, Bahrain)")
    print(f"  • Needs attention: Conflict-affected countries")

print("\n📋 INDICATORS AVAILABLE:")
for code, name in key_indicators.items():
    print(f"  • {code}: {name}")

print("\n💡 USAGE EXAMPLES:")
print("  who = WHOAccessor()")
print("  emro_data = who.get_emro_countries()")
print("  hale = who.get_emro_life_expectancy(years=[2020,2021])")
print("  malaria = who.get_emro_malaria_data(years=[2019,2020])")
print("  trends = who.get_emro_health_trends('WHOSIS_000001', 2015, 2021)")

print("\n✅ EMRO analysis complete!")
print("="*70)

## 11. Export Data

In [ ]:
# Export EMRO data for further analysis
import os

export_path = "./emro_analysis_export/"
os.makedirs(export_path, exist_ok=True)

# Export country list
emro_countries.to_csv(f"{export_path}emro_countries.csv", index=False)
print(f"✅ Exported EMRO countries to {export_path}emro_countries.csv")

# Export HALE data if available
if not hale_data.empty:
    hale_data.to_csv(f"{export_path}emro_hale_data.csv", index=False)
    print(f"✅ Exported HALE data to {export_path}emro_hale_data.csv")

# Export malaria data if available
if 'malaria_data' in locals() and not malaria_data.empty:
    malaria_data.to_csv(f"{export_path}emro_malaria_data.csv", index=False)
    print(f"✅ Exported malaria data to {export_path}emro_malaria_data.csv")

print(f"\n📁 All exports saved to: {os.path.abspath(export_path)}")